In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [2]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [3]:
import json

def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    response = chat(messages, stop_sequences=["```"])
    return json.loads(response)

In [ ]:
dataset = generate_dataset()
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [11]:
def grade_by_model(test_case, output):
    prompt = f"""
    You are an expert AWS code reviewer. Your task is to evaluate the following AI generated solution:
    
    Original Task:
    <task>
    {test_case["task"]}
    </task>
    
    Solution to Evaluate:
    <solution>
    {output}
    </solution>
    
    Output Format
    Provide your evaluation as a structured JSON object with the following format:
    - "strengths": A list of strengths of the solution.
    - "weaknesses": An array of 1-3 areas for improvement.
    - "reasoning": A detailed explanation of your evaluation, including why you assigned the score.
    - "score": An overall score for the solution on a scale of 1 to 10.
    
    Respond with only the JSON object, without any additional text or explanation.
    Example output:
    {{
        "strengths": ["Strength 1", "Strength 2"],
        "weaknesses": ["Weakness 1", "Weakness 2"], 
        "reasoning": "Detailed explanation of the evaluation and reasoning behind the score.",
        "score": 7
    }}
    """
    
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    response = chat(messages, stop_sequences=["```"])
    return json.loads(response)

def run_prompt(test_case):
    # Merges the prompt and test case input, then returns the result
    prompt = f"""
    Please solve the following task:
    
    {test_case["task"]}"""
    
    messages = []
    add_user_message(messages, prompt)
    response = chat(messages)
    return response

def run_test_case(test_case):
    output = run_prompt(test_case)
    
    # todo grading
    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }
    
from statistics import mean
    
def run_eval(dataset):
    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
        
    average_score = mean([result["score"] for result in results])
    print(f"Average Score: {average_score}")
    return results

In [9]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)
    
results = run_eval(dataset)

In [ ]:
print(json.dumps(results, indent=2))

Average Score: 7.333333333333333
[
  {
    "output": "# AWS S3 Region Extractor\n\nHere's a Python function that extracts the AWS region from an S3 bucket URL:\n\n```python\nimport re\nfrom urllib.parse import urlparse\n\ndef extract_region_from_s3_url(s3_url: str) -> str:\n    \"\"\"\n    Extracts the AWS region from an S3 bucket URL.\n    \n    Supports formats:\n    - s3://bucket-name.region.amazonaws.com\n    - s3://bucket-name-region.amazonaws.com\n    - https://bucket-name.s3.region.amazonaws.com\n    - https://s3.region.amazonaws.com/bucket-name\n    \n    Args:\n        s3_url: The S3 bucket URL string\n        \n    Returns:\n        The AWS region string (e.g., 'us-east-1')\n        \n    Raises:\n        ValueError: If no region can be extracted from the URL\n    \"\"\"\n    \n    # Pattern 1: s3://bucket-name.region.amazonaws.com\n    pattern1 = r's3://[^.]+\\.([a-z0-9\\-]+)\\.amazonaws\\.com'\n    match = re.search(pattern1, s3_url)\n    if match:\n        return match.gro